# Systematic multi-tool benchmark

This notebook compares ViralSafeTarget with compatible guide-design and off-target tools on a frozen panel. It separates executable results from documented capabilities. Missing exports remain `pending` or `export_required`; they are never interpreted as zero predicted hits. The analysis makes no claim about editing efficacy, safety, viral inactivation, treatment, or cure.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
from viral_safe_target.notebook_helpers import find_project_root
from viral_safe_target.tool_benchmark import run_ablation
try:
    from IPython.display import display
except ImportError:
    display = print

root = find_project_root()
MODE = os.environ.get('VST_NOTEBOOK_MODE', 'synthetic').lower()
assert MODE in {'synthetic', 'real'}
print(f'Mode: {MODE}')

## Comparison theory

CRISPOR and CHOPCHOP are primarily guide-design/ranking systems; CRISPRitz, GuideScan2, and Cas-OFFinder emphasize different forms of off-target enumeration or specificity. Raw scores are therefore not averaged. The benchmark compares within-tool ranks, top-K overlap, candidate coverage, missingness, and documented scope. Biological superiority requires independent experimental ground truth.

In [ ]:
if MODE == 'synthetic':
    candidates = pd.DataFrame({
        'candidate_id': ['g1', 'g2', 'g3', 'g4'],
        'conservation_score': [1.0, 0.9, 0.8, 0.7],
        'viral_uniqueness_score': [0.5, 1.0, 0.8, 0.9],
        'penalty': [0.0, 0.0, 0.0, 0.1],
    })
    _, ablation = run_ablation(
        candidates,
        [
            {'name': 'conservation', 'column': 'conservation_score', 'weight': 0.7},
            {'name': 'uniqueness', 'column': 'viral_uniqueness_score', 'weight': 0.3},
        ],
        ['penalty'],
        [2],
    )
    display(ablation)
else:
    snapshot = root / 'reports/hsv2_tool_benchmark'
    manifest = json.loads((snapshot / 'run_manifest.json').read_text())
    status = pd.read_csv(snapshot / 'tool_execution_status.csv')
    agreement = pd.read_csv(snapshot / 'rank_agreement.csv')
    overlap = pd.read_csv(snapshot / 'top_k_overlap.csv')
    ablation = pd.read_csv(snapshot / 'ablation_summary.csv')
    assert manifest['candidate_count'] == 257
    assert manifest['unique_guide_count'] == 257
    assert not status['status'].isna().any()
    display(status)


## Auditable real-data outputs

In real mode, the committed snapshot verifies the 257-guide identity set, tool versions, raw-output hashes, execution status, rank agreement, top-K overlap, and ablation sensitivity. A capability table is presented separately and cites official papers, documentation, or repositories.

In [ ]:
if MODE == 'real':
    display(agreement)
    display(overlap.head(30))
    display(ablation)
    print('Completed tools:', ', '.join(manifest['completed_tools']))
    print('Incomplete tools:', ', '.join(manifest['incomplete_tools']))
else:
    print('Synthetic mode validates the analysis logic without claiming a real tool comparison.')

## Limitations

- A missing web-tool export is not a negative result.
- Scores from different tools do not share a common biological scale.
- Runtime is comparable only when hardware, genome, search model, and guide count are identical.
- Capability evidence describes documented scope, not predictive performance.
- Leave-one-component-out analysis is sensitivity analysis, not retraining.
- Experimental ground truth and second-virus validation remain necessary for stronger claims.